In [ ]:
import pandas as pd 
import numpy as np
import gspread as gs
from gspread_dataframe import set_with_dataframe
from gspread_formatting import CellFormat, Color, set_frozen, set_column_width, format_cell_range

In [3]:
prime_dt = pd.read_csv("./LayerSC/Layer10.csv")
# print(prime_dt,'\n')

mean = prime_dt.mean()  
sum = prime_dt.sum()   
std_dev = prime_dt.loc[:, prime_dt.columns != 'W_content'].std(ddof=1)
variance = prime_dt.loc[:, prime_dt.columns != 'W_content'].var(ddof=1)

# print(prime_dt.columns)
print('Standard Deviation:--------------\n',np.round(std_dev, 4), '\n')
print('Variance:--------------\n',np.round(variance, 4), '\n')
print('Mean:--------------\n',np.round(mean, 2), '\n')
print('Sum:--------------\n',sum, '\n')

Standard Deviation:--------------
 Wet_U_weight    0.0240
Dry_U_weight    0.0310
Sub_U_weight    0.0191
dtype: float64 

Variance:--------------
 Wet_U_weight    0.0006
Dry_U_weight    0.0010
Sub_U_weight    0.0004
dtype: float64 

Mean:--------------
 W_content       19.29
Wet_U_weight     1.97
Dry_U_weight     1.65
Sub_U_weight     1.03
dtype: float64 

Sum:--------------
 W_content       771.80
Wet_U_weight     78.81
Dry_U_weight     66.09
Sub_U_weight     41.27
dtype: float64 



In [33]:
Wet_unit_weight_dt = prime_dt['Wet_U_weight']
Wet_unit_weight_dt.index.name = None
Wet_unit_weight_dt = Wet_unit_weight_dt.to_frame()

Wet_unit_weight_var = Wet_unit_weight_dt['Wet_U_weight'].var()
print('OK \n' if Wet_unit_weight_var < 0.05 else 'Failed')

Wet_unit_weight_count = Wet_unit_weight_dt['Wet_U_weight'].count()
print('count:',Wet_unit_weight_count,'\n')

Wet_unit_weight_mean = Wet_unit_weight_dt['Wet_U_weight'].mean()
print('mean:',Wet_unit_weight_mean,'\n')

Sample_pass_cond = Wet_unit_weight_dt['Wet_U_weight'].std()
print('[v]=', Sample_pass_cond, '\n') 

Wet_unit_weight_dt['Ad'] = (Wet_unit_weight_dt['Wet_U_weight'] - Wet_unit_weight_mean).abs()

Wet_unit_weight_dt['Check'] = Wet_unit_weight_dt['Ad'] < Sample_pass_cond


Wuw_dt_clean = Wet_unit_weight_dt.loc[Wet_unit_weight_dt['Check'] == True]
Wuw_dt_clean.Name = 'Clean_data'
characteristic_value = Wet_unit_weight_dt['Wet_U_weight'].mean()    
print('characteristic value:', np.round(characteristic_value, 2), '\n')

# print(Wet_unit_weight_dt.loc[Wet_unit_weight_dt['Check'] == True])

# Wet_unit_weight_dt['check'] 

OK 

count: 40 

mean: 1.97025 

[v]= 0.024017888632412596 

characteristic value: 1.97 



## Calculate limit state design values

In [ ]:
## ultimate limit state design value:
t_coef_data = pd.read_csv("./Coefficients/t_coef_sheet.csv")

t_1 = t_coef_data[t_coef_data['n'] == 40]['0.95'].values[0]
t_2 = t_coef_data[t_coef_data['n'] == 40]['0.85'].values[0]
n_index = Wet_unit_weight_count - 1

print('Ultimate limit state design value t (TTGH I): ',t_1)
print('Serviceability limit state design value t (TTGH II): ',t_2)

rho_ultimate =  (t_1 - Wet_unit_weight_var) / np.sqrt(n_index)
rho_serviceability = (t_2 - Wet_unit_weight_var) / np.sqrt(n_index)

print('rho_ultimate:', rho_ultimate)
print('rho_serviceability:', rho_serviceability)
print('---------------------------------------------\n')

print(f'gamma_I = {characteristic_value:.2f}(1 ± {rho_ultimate:.4f})')
print(f'gamma_II = {characteristic_value:.2f}(1 ± {rho_serviceability:.4f})')


ultimate limit state design value t (TTGH I):  1.68
serviceability limit state design value t (TTGH II):  1.05
rho_ultimate: 0.2689229270299764
rho_serviceability: 0.16804219013277152
---------------------------------------------

gamma_I = 1.97(1 ± 0.2689)
gamma_II = 1.97(1 ± 0.1680)


### optional
this code block is used for converting dataframe to a spreadsheet for a Google sheet file.

In [5]:

# gc = gs.service_account(filename='pysheetAuth.json')

# # v_coef = gc.open('Pysheet').worksheet('v_coef')
# # v_coef_data = v_coef.get_all_records()
# # v_coef_sheet = pd.DataFrame(v_coef_data)
# # v_coef_sheet.to_csv('v_coef_sheet.csv', index=False)
# # print(v_coef_sheet)

# t_coef = gc.open('Pysheet').worksheet('t_coef')
# t_coef_data = t_coef.get_all_records()
# t_coef_sheet = pd.DataFrame(t_coef_data)
# t_coef_sheet.to_csv('./Coefficients/t_coef_sheet.csv', index=False)
# print(t_coef_sheet)

# # # Update the sheet with the Wuw_dt_clean dataframe
# # # Define the cell format for the header
# # header_format = CellFormat(
# #     backgroundColor=Color(0, 1, 0),  # Green color
# #     textFormat={'bold': True, 'fontSize': 12}
# # )

# # # Update the sheet with the Wuw_dt_clean dataframe
# # set_with_dataframe(sheet, Wuw_dt_clean)

# # # Apply the header format to the first row
# # format_cell_range(sheet, 'A1:C1', header_format)
